In [1]:
import json
import random
import os

# Path to your original JSON
INPUT_PATH = "msasl_top100_splits/all_top100.json"

# Output file paths
TRAIN_PATH = "msasl_top100_splits/train.json"
VAL_PATH   = "msasl_top100_splits/val.json"
TEST_PATH  = "msasl_top100_splits/test.json"

# Desired split ratios
TRAIN_RATIO = 0.75
VAL_RATIO   = 0.10
TEST_RATIO  = 0.15

# Make sure ratios sum to 1.0
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-6

# 1) Load the full list of samples
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    all_data = json.load(f)

# 2) Shuffle in place (to randomize which go into train/val/test)
random.shuffle(all_data)

N = len(all_data)
n_train = int(N * TRAIN_RATIO)
n_val   = int(N * VAL_RATIO)
# Ensure test gets the remainder
n_test  = N - n_train - n_val

train_data = all_data[:n_train]
val_data   = all_data[n_train : n_train + n_val]
test_data  = all_data[n_train + n_val :]

print(f"Total samples: {N}")
print(f" →   Train: {len(train_data)}")
print(f" →     Val: {len(val_data)}")
print(f" →    Test: {len(test_data)}")

# 3) Write each split out to its own file
os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)

with open(TRAIN_PATH, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(VAL_PATH, "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

with open(TEST_PATH, "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

print("Wrote train.json, val.json, test.json")

Total samples: 3949
 →   Train: 2961
 →     Val: 394
 →    Test: 594
Wrote train.json, val.json, test.json


In [2]:
import json
import numpy as np

# 1) Adjust this path if necessary to where your JSON actually resides
INPUT_PATH = "msasl_top100_splits/all_top100.json"

# 2) Load the full dataset
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# 3) Compute number of frames per clip
#    Using: frame_count = round((end_time - start_time) * fps)
frame_counts = []
for entry in data:
    fps = entry.get("fps", 0)
    start_time = entry.get("start_time", 0)
    end_time = entry.get("end_time", 0)
    duration = end_time - start_time
    frame_count = int(round(duration * fps))
    frame_counts.append(frame_count)

frame_counts = np.array(frame_counts)

# 4) Calculate statistics
min_frames = frame_counts.min()
max_frames = frame_counts.max()
avg_frames = frame_counts.mean()

print(f"Minimum frames   : {min_frames}")
print(f"Maximum frames   : {max_frames}")
print(f"Average frames   : {avg_frames:.2f}")

Minimum frames   : 0
Maximum frames   : 290
Average frames   : 85.30


In [3]:
import json

INPUT_PATH = "msasl_top100_splits/all_top100.json"
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

zero_frame_entries = []
for entry in data:
    fps = entry.get("fps", 0)
    start = entry.get("start_time", 0)
    end   = entry.get("end_time", 0)
    # Compute raw frame count, without rounding up too aggressively
    raw_frames = (end - start) * fps
    if round(raw_frames) == 0:
        zero_frame_entries.append({
            "file":       entry.get("file"),
            "start_time": start,
            "end_time":   end,
            "fps":        fps,
            "duration":   end - start
        })

print(f"Found {len(zero_frame_entries)} entries with 0 frames:")
for e in zero_frame_entries:
    print(e)

Found 1 entries with 0 frames:
{'file': 'In the Classroom', 'start_time': 133.7, 'end_time': 133.7, 'fps': 29.97, 'duration': 0.0}


In [ ]:
import json

# 1. Load the JSON file
with open('train_keypoints.json', 'r', encoding='utf-8') as f:
    data_list = json.load(f)

# 2. Collect all processed_frames values
processed_frames = []
for item in data_list:
    try:
        frames = item["keypoints_data"]["video_info"]["processed_frames"]
        processed_frames.append(frames)
    except KeyError:
        print(f"Warning: 'processed_frames' not found in item {item.get('original_data', {}).get('file', 'unnamed')}")

# 3. Calculate statistics
if processed_frames:
    max_val = max(processed_frames)
    min_val = min(processed_frames)
    avg_val = sum(processed_frames) / len(processed_frames)
    print(f"Maximum processed_frames: {max_val}")
    print(f"Minimum processed_frames: {min_val}")
    print(f"Average processed_frames: {avg_val:.2f}")
else:
    print("No valid processed_frames values found.")
